In [2]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn glob2 pyarrow matplotlib seaborn

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/27.4 MB ? eta -:--:--
   ---------------- ----------------------- 11.5/27.4 MB 51.5 MB/s eta 0:00:01
   ---------------------------------------  27.3/27.4 MB 69.2 MB/s eta 0:00:01
   ---------------------------------------- 27.4/27.4 MB 57.8 MB/s  0:00:00
  Created wheel for glob2: filename=glob2-0.7-py2.py3-none-any.whl size=9378 sha256=4b62fa968c85826930c002c0aae027bc6d648c4571af9ac83b2a959d25a9d6dc
  Stored in directory: c:\users\loant\appdata\local\pip\cache\wheels\fb\b3\7b\71827dcd17a71be7a8b7adf9f30a4200b612a903d7d8af3442
Successfully built

In [3]:
import pandas as pd
import glob
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report

# ==========================================
# LE THANH DAT (TASK 2.2)
# 1. TỔNG HỢP VÀ DỌN DẸP DỮ LIỆU 
# ==========================================

path = '../data/*.csv'
files = glob.glob(path)

# Nếu đã có file pkl rồi thì load luôn, không cần đọc lại 8 CSV
if os.path.exists('../models/cleaned_data.pkl'):
    print("Tìm thấy cleaned_data.pkl, đang load...")
    df = joblib.load('../models/cleaned_data.pkl')
    print("Load xong!")

elif not files:
    print("LỖI: Không tìm thấy file CSV nào trong thư mục data!")

else:
    # Gộp tất cả file thành 1 DataFrame lớn
    print(f"Đang gộp {len(files)} file dữ liệu...")
    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    # 1. Dọn dẹp khoảng trắng tên cột
    df.columns = df.columns.str.strip()

    # Mapping từ tên cột thực tế (của bạn) sang tên cột yêu cầu (của Lab)
    column_mapping = {
        'Destination Port': 'Dst Port',
        'Total Fwd Packets': 'Tot Fwd Pkts',
        'Total Backward Packets': 'Tot Bwd Pkts',
        'Total Length of Fwd Packets': 'TotLen Fwd Pkts',
        'Total Length of Bwd Packets': 'TotLen Bwd Pkts',
        'Fwd Packet Length Mean': 'Fwd Pkt Len Mean',
        'Bwd Packet Length Mean': 'Bwd Pkt Len Mean',
        'Flow Bytes/s': 'Flow Byts/s',
        'Flow Packets/s': 'Flow Pkts/s',
        'Packet Length Mean': 'Pkt Len Mean',
        'Packet Length Std': 'Pkt Len Std',
        'FIN Flag Count': 'FIN Flag Cnt',
        'SYN Flag Count': 'SYN Flag Cnt',
        'RST Flag Count': 'RST Flag Cnt',
        'PSH Flag Count': 'PSH Flag Cnt',
        'ACK Flag Count': 'ACK Flag Cnt',
        'URG Flag Count': 'URG Flag Cnt'
    }

    df.rename(columns=column_mapping, inplace=True)
    print("Đã đồng bộ hóa tên cột theo yêu cầu Lab.")

    # 2. Loại bỏ dữ liệu trùng lặp trước (Giảm số hàng cần xử lý phía sau)
    print("Đang loại bỏ dữ liệu trùng lặp...")
    df.drop_duplicates(inplace=True)

    # 3. Xử lý giá trị vô hạn (inf)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 4. Điền các giá trị trống bằng trung vị (median)
    # Lưu ý: numeric_only=True là bắt buộc để tránh lỗi với cột Label (chữ)
    df.fillna(df.median(numeric_only=True), inplace=True)

    # 5. Loại bỏ các cột biến thiên bằng 0 (PHẢI GÁN LẠI BIẾN DF)
    print("Đang loại bỏ các cột zero-variance...")
    non_zero_var_cols = [col for col in df.columns if df[col].nunique() > 1]
    df = df[non_zero_var_cols] # <--- Cần dòng này để thực thi việc lọc

    # 6. Tối ưu RAM
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype('float32')

    # Lưu lại
    joblib.dump(df, '../models/cleaned_data.pkl')
    print(f"Hoàn thành! Số lượng cột còn lại: {df.shape[1]}")

Đang gộp 8 file dữ liệu...
Hoàn thành sơ chế dữ liệu và đã lưu cleaned_data.pkl!


In [4]:
# ==========================================
# NGUYEN QUOC DAT 
# 2.3 XỬ LÝ MẤT CÂN BẰNG DỮ LIỆU
# ==========================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import joblib

# Load df nếu chưa có
if 'df' not in dir():
    df = joblib.load('../models/cleaned_data.pkl')

# Encode Label
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])
print("Các nhãn:", list(le.classes_))

# Tách X (78 features), y
X = df.drop('Label', axis=1)
y = df['Label']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Xem phân phối trước
print("\nTrước khi balance:")
print(pd.Series(y_train).value_counts())

# SMOTE + RandomUnderSampler
majority_count = pd.Series(y_train).value_counts().max()
threshold = int(majority_count * 0.1)

sampling_strategy_smote = {
    cls: max(count, threshold)
    for cls, count in pd.Series(y_train).value_counts().items()
    if count < majority_count
}

pipeline = Pipeline([
    ('smote', SMOTE(sampling_strategy=sampling_strategy_smote, random_state=42)),
    ('under', RandomUnderSampler(sampling_strategy='majority', random_state=42))
])

X_train, y_train = pipeline.fit_resample(X_train, y_train)

print("\nSau khi balance:")
print(pd.Series(y_train).value_counts())

# Lưu lại
joblib.dump((X_train, X_test, y_train, y_test), '../models/balanced_data.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le, '../models/label_encoder.pkl')
print("\nĐã lưu balanced_data.pkl!")

ModuleNotFoundError: No module named 'imblearn'

In [6]:
print(X.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

In [10]:
# ==========================================
# LE KIM BUU (TASK 2.4)
# 2.4 FEATURE SELECTION
# ==========================================

# lưu lại tên cột gốc
feature_names = X.columns

# convert numpy → DataFrame
X_train = pd.DataFrame(X_train, columns=feature_names)
X_test = pd.DataFrame(X_test, columns=feature_names)

# chọn feature đúng theo dataset của bạn
selected_features = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Total Length of Fwd Packets',
    'Total Length of Bwd Packets',
    'Fwd Packet Length Mean',
    'Bwd Packet Length Mean',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Packet Length Mean',
    'Packet Length Std',
    'SYN Flag Count',
    'ACK Flag Count',
    'FIN Flag Count',
    'RST Flag Count',
    'PSH Flag Count',
    'URG Flag Count'
]

# check lỗi trước khi select
missing = set(selected_features) - set(X_train.columns)
print("Thiếu:", missing)

# select feature
X_train = X_train[selected_features]
X_test = X_test[selected_features]

print("\nSau khi feature selection:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Thiếu: set()

Sau khi feature selection:
X_train: (2017889, 17)
X_test: (504473, 17)


In [8]:
# ==========================================
# LE THANH DAT (TASK 2.5)
# 2. HUẤN LUYỆN MÔ HÌNH BASELINE 
# ==========================================

# Mã hóa nhãn (Chuyển tên cuộc tấn công thành số)
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])

# Chia đặc trưng (X) và nhãn mục tiêu (y)
X = df.drop('Label', axis=1)
y = df['Label']

# Chia tập Train (80%) và Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Chuẩn hóa dữ liệu (Scaling) - Cực kỳ quan trọng cho Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Huấn luyện mô hình Logistic Regression
# Tăng max_iter để đảm bảo mô hình hội tụ tốt hơn
print("Đang huấn luyện mô hình Logistic Regression (có thể mất vài phút)...")
model = LogisticRegression(max_iter=500, solver='lbfgs')
model.fit(X_train_scaled, y_train)

# ==========================================
# 3. ĐÁNH GIÁ VÀ LƯU TRỮ (OUTPUT)
# ==========================================

# Tính toán độ chính xác tổng quát
accuracy = model.score(X_test_scaled, y_test)
print(f"\n--- KẾT QUẢ ---")
print(f"Độ chính xác tổng thể: {accuracy:.4f}")

# Dự đoán và in báo cáo chi tiết (Precision, Recall, F1-score)
y_pred = model.predict(X_test_scaled)
print("\nBáo cáo chi tiết cho từng loại tấn công:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Tạo thư mục và lưu trữ model đã học
joblib.dump(model, '../models/logistic_regression.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print("\nĐã lưu 'logistic_regression.pkl' và 'scaler.pkl' vào thư mục models.")

Đang huấn luyện mô hình Logistic Regression (có thể mất vài phút)...


KeyboardInterrupt: 